# Few-Shot Object Localization — Colab Training

End-to-end pipeline: **train → evaluate → export to TFLite**.

## Setup checklist

1. **Runtime → Change runtime type → GPU.** T4 (free) is fine; A100 is ~10× faster.
2. Edit `REPO_URL` and `RUN_NAME` in the configuration cell.
3. Run cells top to bottom.

The cleaned dataset is downloaded from a public Drive folder via `gdown`. Checkpoints are written to `MyDrive/iss_group_24/models/<RUN_NAME>/` so they survive runtime disconnects.

## 1. GPU check

In [ ]:
!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. Configuration

Edit `REPO_URL` and `RUN_NAME`, then run.

In [ ]:
REPO_URL          = "https://github.com/pewpewnor/iss_group_24.git"  # change me
RUN_NAME          = "run_001"                                          # bump per training run
DATASET_FOLDER_ID = "1fWmrau8gxeCZT40EQBR-zMTEu42MzC56"               # public 'cleaned' folder on Drive

REPO_DIR     = "/content/iss_group_24"
DATASET_DIR  = f"{REPO_DIR}/dataset/cleaned"
DRIVE_ROOT   = "/content/drive/MyDrive/iss_group_24"
MODELS_DIR   = f"{DRIVE_ROOT}/models/{RUN_NAME}"

print("repo:      ", REPO_URL)
print("dataset:   ", DATASET_DIR)
print("models out:", MODELS_DIR)

## 3. Mount Google Drive

Drive is used only as a persistent destination for checkpoints.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Clone / update repo

In [ ]:
import os, sys, subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!git log -1 --oneline

## 5. Install dependencies

Colab ships with compatible `torch`/`torchvision`. `litert-torch` is needed for the export step only; `gdown` for the dataset download.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q gdown litert-torch

## 6. Download cleaned dataset

Pulls the `cleaned/` folder (manifest, stats, splits) from Drive into the repo's `dataset/cleaned/` path so the default manifest path resolves correctly.

In [ ]:
import os, time
import gdown

if os.path.isfile(f"{DATASET_DIR}/manifest.json"):
    print(f"{DATASET_DIR} already populated \u2014 skipping download")
else:
    os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
    t0 = time.time()
    gdown.download_folder(
        id=DATASET_FOLDER_ID,
        output=DATASET_DIR,
        quiet=False,
        use_cookies=False,
    )
    print(f"  done in {time.time() - t0:.1f}s")

!ls $DATASET_DIR
!cat $DATASET_DIR/stats.json

## 7. Train

Runs Stage 1 (warmup, 8 epochs) then Stage 2 (partial unfreeze, 20 epochs).
Saves `best.pt` (by val IoU) and `last.pt` to Drive after every epoch.

In [ ]:
import os
from modeling.train import train

os.makedirs(MODELS_DIR, exist_ok=True)

train(
    manifest=f"{DATASET_DIR}/manifest.json",
    train_split="train",
    val_split="val",
    out_dir=MODELS_DIR,
    batch_size=16,
    num_workers=2,
    episodes_per_epoch=1000,
    val_episodes=200,
    stage1_epochs=8,
    stage2_epochs=20,
)

## 8. Evaluate

Runs 1000 episodic evaluations on the held-out test split.
Writes `eval_report.json` next to the checkpoint.

In [ ]:
from modeling.evaluate import run as evaluate_run

evaluate_run(
    checkpoint=f"{MODELS_DIR}/best.pt",
    manifest=f"{DATASET_DIR}/manifest.json",
    split="test",
    report=f"{MODELS_DIR}/eval_report.json",
    episodes=1000,
    batch_size=16,
    num_workers=2,
)

## 9. Export to TFLite

Converts `best.pt` to a `.tflite` flatbuffer via `litert_torch`.
INT8 dynamic-range quantization is applied by default (~4 MB output).
Pass `quantize=False` to keep float32 weights (~13 MB).

Model inputs/outputs after export:
- **inputs:** `support_imgs (1,5,3,224,224)`, `support_bboxes (1,5,4)`, `query_img (1,3,224,224)`
- **outputs:** `bbox (1,4)` xyxy in 224-px coords, `score (1,1)` confidence

In [ ]:
from modeling.export import export

export(
    checkpoint=f"{MODELS_DIR}/best.pt",
    out=f"{MODELS_DIR}/model.tflite",
)

## 10. (Optional) Stage 3 fine-tune

Only run if Stage 2 val IoU was still clearly improving at the end.
Resumes from `best.pt` and runs 10 more epochs with the full backbone unfrozen at very low LR.
Re-evaluate after to check improvement.

In [ ]:
# from modeling.train import train
#
# train(
#     manifest=f"{DATASET_DIR}/manifest.json",
#     out_dir=MODELS_DIR,
#     resume=f"{MODELS_DIR}/best.pt",
#     batch_size=16,
#     num_workers=2,
#     episodes_per_epoch=1000,
#     val_episodes=200,
#     stage1_epochs=0,
#     stage2_epochs=0,
#     stage3_epochs=10,
#     stage3=True,
# )